# 04. 모델 평가와 인계 검증

최종 Test와 재사용 Test 진단은 구분합니다. 패키지 smoke 검증은 품질 승인이나 목표 CPU 성능 검증이 아닙니다.

실행 결과는 저장하지 않았습니다. 아래 실행 스위치는 기본 `False`이며, 전체 실행만으로 학습·파일 생성·서버 검사가 시작되지 않습니다.

In [ ]:
from pathlib import Path
import os
import sys
import json

# 저장소 루트 또는 notebooks 폴더에서 실행할 수 있습니다.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src/data').is_dir() and (p / 'configs').is_dir()), None)
if ROOT is None:
    raise RuntimeError('CQC 저장소 안에서 노트북을 열어 주세요.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('작업 폴더:', ROOT)


## 최종 학습·평가 절차

최종 학습/최초 Test는 공통 CLI에 잠금과 확인 절차가 있습니다. 여기서는 자동 호출하지 않습니다. 확정 계획은 src/training/__init__.py 설명을 참고하세요.

In [ ]:
from src.training import final_fit, evaluate, model_card, package_model

# 승인된 설정에 맞는 인자를 먼저 확인합니다. --help는 학습/평가하지 않습니다.
RUN_SHOW_HELP = False
if RUN_SHOW_HELP:
    for module in (final_fit, evaluate, model_card, package_model):
        try:
            module.main(['--help'])
        except SystemExit as exc:
            if exc.code != 0:
                raise


## 재사용 Test 회귀 진단

이미 사용한 27개 사과를 재확인합니다. 모델 선택·최종 승인에 사용하지 않습니다.

In [ ]:
"""이미 사용한 시험 사과 27개의 회귀 진단을 수행한다.

이 결과는 독립 최종 승인이나 모델 선택 점수로 사용하지 않는다.
"""

from __future__ import annotations

from pathlib import Path

import torch
from torch.utils.data import DataLoader

from src.data.multiview import MultiViewDataset, load_groups
from src.data.torch_dataset import TorchMultiViewDataset
from src.data.virtual_brix import load_virtual_brix
from src.training.engine import run_epoch
from src.training.evaluate import load_checkpoint
from src.training.models import build_model
from src.training.train import resolve_device, write_json


In [ ]:
def evaluate_reused_test(*, checkpoint, output, device='auto', manifest=Path('data/processed/manifest.csv'), splits=Path('configs/splits/seed-42.csv'), raw_root=Path('data/raw'), virtual_brix=None):
    """이미 사용한 시험 사과의 회귀 진단을 수행하고 독립 승인과 구분해 기록한다."""
    if output.exists():
        raise ValueError(f"기존 평가 파일을 덮어쓰지 않습니다: {output}")
    device = resolve_device(device)
    checkpoint_path = checkpoint
    checkpoint = load_checkpoint(checkpoint_path, device)
    config = checkpoint["config"]
    if (config["model_kind"] == "separate_brix") != (virtual_brix is not None):
        raise ValueError("separate_brix requires virtual_brix; image-only models must omit it")
    virtual_brix_path = virtual_brix
    virtual_brix = load_virtual_brix(virtual_brix_path) if virtual_brix_path else None
    source = MultiViewDataset(
        load_groups(manifest, splits), raw_root,
        int(config["views"]), split="test",
    )
    loader = DataLoader(
        TorchMultiViewDataset(source, training=False, image_size=int(config["image_size"]), virtual_brix=virtual_brix),
        batch_size=2, shuffle=False, num_workers=0,
    )
    model = build_model(str(config["model_kind"]), pretrained=False).to(device)
    model.load_state_dict(checkpoint["model_state"])
    try:
        metrics = run_epoch(model, loader, device, include_predictions=True)
    finally:
        source.close()
    write_json(output, {
        "evaluation_role": "reused_test_regression_diagnostic",
        "independent_final_approval": False,
        "may_select_model_or_epoch": False,
        "prior_test_usage": "v1 final evaluation and failure analysis",
        "checkpoint": str(checkpoint_path),
        "checkpoint_epoch": checkpoint["epoch"],
        "virtual_brix": str(virtual_brix_path) if virtual_brix_path else None,
        "brix_is_measured": False if virtual_brix else None,
        "config": config,
        "test": metrics,
    })
    print(f"regression_result={output} samples={metrics['samples']} "
          f"quality_macro_f1={metrics['quality']['macro_f1']:.6f}", flush=True)
    return 0


In [ ]:
RUN_EVALUATE_REUSED_TEST = False
if RUN_EVALUATE_REUSED_TEST:
    evaluate_reused_test(checkpoint=Path("outputs/candidate/best.pt"), output=Path("outputs/notebook-reused-test.json"), device="cpu")


## 후보 패키지 검증

가짜 PNG 12장으로 SHA와 API 연결만 검사합니다.

In [ ]:
"""패키지 무결성과 API 연결을 검사한다. 모델 품질이나 네트워크 성능 검사는 별도다."""

from __future__ import annotations

import json
from io import BytesIO
from pathlib import Path

import torch
from fastapi.testclient import TestClient
from PIL import Image

from src.inference.api import create_app
from src.inference.predictor import Predictor


In [ ]:
def verify_candidate_package(*, package, output):
    """가짜 사진 12장으로 패키지·상태 확인·예측 API의 연결을 검사한다."""
    if output.exists():
        raise ValueError("Existing verification report must not be overwritten")
    torch.set_num_threads(1)
    predictor = Predictor(package, device="cpu")
    buffer = BytesIO()
    Image.new("RGB", (224, 224), (160, 40, 30)).save(buffer, format="PNG")
    metadata = [dict(view_index=i, angle_direction="top" if i < 6 else "bottom",
                     verticality_angle=45, horizontality_angle=(i % 6) * 60)
                for i in range(12)]
    with TestClient(create_app(predictor)) as client:
        health = client.get("/health")
        response = client.post("/v1/predict", data={
            "inspection_id": "candidate-smoke-12", "metadata": json.dumps(metadata),
        }, files=[("images", (f"{i}.png", buffer.getvalue(), "image/png")) for i in range(12)])
    if health.status_code != 200 or response.status_code != 200:
        raise RuntimeError(f"API verification failed: {health.text}, {response.text}")
    prediction = response.json()
    if prediction["inspection_id"] != "candidate-smoke-12" or prediction["used_frame_count"] != 12:
        raise RuntimeError("Inspection ID or frame count mismatch")
    report = {
        "scope": "synthetic_image_inprocess_api_smoke_only",
        "quality_evaluated": False, "production_approved": False,
        "target_cpu_verified": False, "test_used": False,
        "checkpoint_sha256": predictor.manifest["checkpoint_sha256"],
        "health": health.json(), "prediction": prediction,
    }
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(report))


In [ ]:
RUN_VERIFY_CANDIDATE_PACKAGE = False
if RUN_VERIFY_CANDIDATE_PACKAGE:
    verify_candidate_package(package=Path("models/cqc-apple-separate12-focal-v2-candidate"), output=Path("outputs/notebook-candidate-smoke.json"))
